# E-commerce Fraud Detection - Exploratory Data Analysis & Preprocessing
This notebook loads the raw e-commerce transaction data, cleans it, maps IP addresses to countries, performs exploratory data analysis, and scales/one-hot encodes the features for modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Append src path
sys.path.append('..')
from src.data_preprocessing import load_data, clean_fraud_data, engineer_features, scale_and_encode


## 1. Load Datasets
We load `Fraud_Data.csv` and the range mapping file `IpAddress_to_Country.csv` using the modular preprocessing module.


In [ ]:
fraud_df, ip_df, _ = load_data('../data')
print("Raw Fraud Data Shape:", fraud_df.shape)
print("Raw IP Mapping Shape:", ip_df.shape)


## 2. Data Cleaning and Geolocation Mapping
We convert raw columns to appropriate types, convert datetime timestamps, cast IP addresses, and merge with the IP-to-country bounds using an optimized backward range-based join. Unmapped IP ranges are marked as `'Unknown'` to retain 100% of our rows (avoiding the loss of ~14.5% data).


In [ ]:
cleaned_fraud = clean_fraud_data(fraud_df, ip_df)
print("Cleaned and Mapped Data Shape:", cleaned_fraud.shape)
print("Missing values:\n", cleaned_fraud.isnull().sum())
print("Duplicates in cleaned data:", cleaned_fraud.duplicated().sum())


## 3. Exploratory Data Analysis
Let's analyze univariate distributions and bivariate relationships with the fraud target class.


In [ ]:
# Fraud class distribution
class_counts = cleaned_fraud["class"].value_counts()
class_pct = cleaned_fraud["class"].value_counts(normalize=True) * 100
print("Class Distribution:")
print(class_counts)
print("\nPercentage:")
print(class_pct)

plt.figure(figsize=(5,4))
sns.countplot(data=cleaned_fraud, x="class", palette='Set2')
plt.title("E-commerce Fraud Class Distribution")
plt.xlabel("Class (0 = Legit, 1 = Fraud)")
plt.ylabel("Count")
plt.show()


In [ ]:
# Age distribution
plt.figure(figsize=(8,5))
sns.histplot(cleaned_fraud["age"], bins=30, kde=True, color='teal')
plt.title("Age Distribution of Users")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()


In [ ]:
# Purchase value distribution
plt.figure(figsize=(8,5))
sns.histplot(cleaned_fraud["purchase_value"], bins=30, kde=True, color='purple')
plt.title("Purchase Value Distribution")
plt.xlabel("Purchase Value ($)")
plt.ylabel("Count")
plt.show()


In [ ]:
# Purchase Value by Class
plt.figure(figsize=(7,5))
sns.boxplot(data=cleaned_fraud, x="class", y="purchase_value", palette='Set3')
plt.title("Purchase Value by Transaction Class")
plt.xlabel("Class (0 = Legit, 1 = Fraud)")
plt.ylabel("Purchase Value ($)")
plt.show()


In [ ]:
# Top 15 Countries by Fraud Rate
country_fraud = cleaned_fraud.groupby("country")["class"].mean().sort_values(ascending=False)
plt.figure(figsize=(12,6))
country_fraud.head(15).plot(kind="bar", color='crimson')
plt.title("Top 15 Countries by Fraud Rate")
plt.ylabel("Fraud Rate")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Feature Engineering and Transformation
We build account age (`time_since_signup` in hours), time features (`hour_of_day`, `day_of_week`), device sharing, and device/IP velocity counters. To prevent one-hot dimensions from blowing up, countries with <50 occurrences are grouped into `'Other'`. We then scale numeric variables and one-hot encode categoricals.


In [ ]:
engineered_fraud = engineer_features(cleaned_fraud)
print("Engineered Data Shape:", engineered_fraud.shape)

# Scale and encode
scaled_fraud, _ = scale_and_encode(engineered_fraud, None)
print("Processed Feature Matrix Shape:", scaled_fraud.shape)

# Save processed data
os.makedirs('../data/processed', exist_ok=True)
scaled_fraud.to_csv('../data/processed/processed_fraud.csv', index=False)
print("Successfully saved processed dataset to ../data/processed/processed_fraud.csv")
